In [1]:
import numpy as np 
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('8-fraud_detection.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   transaction_amount      10000 non-null  float64
 1   transaction_risk_score  10000 non-null  float64
 2   is_fraud                10000 non-null  int64  
dtypes: float64(2), int64(1)
memory usage: 234.5 KB


In [4]:
df.describe()

,transaction_amount,transaction_risk_score,is_fraud
count,10000.000000,10000.000000,10000.000000
mean,0.976419,-1.003136,0.015400
std,0.725346,0.789194,0.123144
min,-3.370100,-3.952121,0.000000
25%,0.505517,-1.538232,0.000000
50%,0.990240,-0.997064,0.000000
75%,1.461125,-0.466221,0.000000
max,3.487193,1.872543,1.000000


In [5]:
df['is_fraud'].value_counts()  # Dengesiz bir dataset

is_fraud
0    9846
1     154
Name: count, dtype: int64

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X = df.drop('is_fraud' , axis = 1)
y = df['is_fraud']

X_train , X_test , y_train , y_test = train_test_split(X , y , random_state = 15 , test_size = 0.25)

In [8]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()

In [9]:
penalty = ['l1' , 'l2' , 'elasticnet']
c_values = [100 , 10 , 0 , 0.1 , 0.01]
solver = ['lbfgs' , 'sag' , 'saga' , 'newton-cholesky' , 'newton-cg' , 'liblinear']
class_weight = [{0 : w , 1 : y} for w in [1 , 10 , 50 , 100] for y in [100 , 50 , 10 , 1]]

In [10]:
params = dict(penalty = penalty , C = c_values , solver = solver , class_weight = class_weight)

In [11]:
from sklearn.model_selection import GridSearchCV , StratifiedKFold

In [12]:
cv = StratifiedKFold()
grid = GridSearchCV(estimator = model , param_grid = params, cv = cv ,scoring = 'accuracy')

In [13]:
import warnings
warnings.filterwarnings('ignore')

In [14]:
grid.fit(X_train , y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
             estimator=LogisticRegression(),
             param_grid={'C': [100, 10, 0, 0.1, 0.01],
                         'class_weight': [{0: 1, 1: 100}, {0: 1, 1: 50},
                                          {0: 1, 1: 10}, {0: 1, 1: 1},
                                          {0: 10, 1: 100}, {0: 10, 1: 50},
                                          {0: 10, 1: 10}, {0: 10, 1: 1},
                                          {0: 50, 1: 100}, {0: 50, 1: 50},
                                          {0: 50, 1: 10}, {0: 50, 1: 1},
                                          {0: 100, 1: 100}, {0: 100, 1: 50},
                                          {0: 100, 1: 10}, {0: 100, 1: 1}],
                         'penalty': ['l1', 'l2', 'elasticnet'],
                         'solver': ['lbfgs', 'sag', 'saga', 'newton-cholesky',
                                    'newton-cg', 'liblinear']},
             scoring='accuracy')

In [15]:
y_pred = grid.predict(X_test)
y_pred

array([0, 0, 0, ..., 0, 0, 0])

In [16]:
from sklearn.metrics import accuracy_score , confusion_matrix , classification_report

In [17]:
score = accuracy_score(y_test , y_pred)
matrix = confusion_matrix(y_test , y_pred)
classification = classification_report(y_test , y_pred)

print('Accuracy Score : ' , score)
print('Classification :\n ' , classification)
print('Matrix : \n' , matrix)

Accuracy Score :  0.99
Classification :
                precision    recall  f1-score   support

           0       0.99      1.00      0.99      2462
           1       0.84      0.42      0.56        38

    accuracy                           0.99      2500
   macro avg       0.92      0.71      0.78      2500
weighted avg       0.99      0.99      0.99      2500

Matrix : 
 [[2459    3]
 [  22   16]]


In [18]:
grid.best_params_

{'C': 0.01, 'class_weight': {0: 10, 1: 50}, 'penalty': 'l2', 'solver': 'sag'}

In [19]:
# ROC - AUC

In [34]:
model_prob = grid.predict_proba(X_test)

In [35]:
model_prob

array([[0.99586418, 0.00413582],
       [0.8529205 , 0.1470795 ],
       [0.95231975, 0.04768025],
       ...,
       [0.98898933, 0.01101067],
       [0.98454492, 0.01545508],
       [0.99758757, 0.00241243]])

In [26]:
model_prob = model_prob[:,1]

In [27]:
from sklearn.metrics import roc_auc_score , roc_curve

In [32]:
model_auc = roc_auc_score(y_test , model_prob)

In [33]:
model_auc

np.float64(0.7392898371029116)